# MouseFull K-Fold training and evaluation

Joint **D1 + D3** cross-validation with eleven anatomical classes, the existing SHAM/TBI classifier and the corrected partial-label ROI objective. Use the **rnet** environment. Inspect the configuration, annotation coverage and animal splits before enabling training.

Each fold separates **training**, **validation for checkpoint selection**, and **independent outer holdout** animals. All scans of an animal stay together; every CV animal is held out exactly once. This is grouped K-Fold with an inner validation split, not a nested hyperparameter search. The shared experiment core performs fitting, checkpoint selection, resumption and full-volume evaluation.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT: Path = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/experiments/config.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.experiments.training import configure_runtime

TRAINING_SEED: int = 42
runtime = configure_runtime(seed=TRAINING_SEED, memory_growth=True, mixed_precision=False)
runtime
import nibabel as nib

REFERENCE: Path = (
    REPO_ROOT / "example" / "RARE" / "TBI_fm_19_49"
    / "Anat" / "TBI_fm_19_49_N4.nii.gz"
)
ref_img = nib.load(REFERENCE)
reference_axes = nib.aff2axcodes(ref_img.affine)
print("Reference axes:", reference_axes, "(left-right is the R/L axis)")


## Complete MouseFull configuration

The vocabulary, channel order, model flags and fitting settings match notebook 05. Contra/ipsilateral anatomy remains separate. Extra anatomy outside the selected mapping is projected to task background. Original network widths and builder dropout behavior remain unchanged. Every fold starts from the selected TBI backbone with a fresh optimizer; output heads are initialized independently for each fold.

Availability follows normalized annotation metadata. Missing extended D1 anatomy and unannotated SHAM lesions receive **-1** targets. `dice_ce_gradient_fix` consumes logits during fitting; exported inference models expose full softmax probabilities. The documented positive-available-target policy does not establish complete background annotation.

Only training applies augmentation and scan repetition. Validation is deterministic and keeps its last incomplete batch. Confirmed duplicate animals across cohorts can be grouped using `animal_ids`; biological identity is not inferred from similar filenames. D2 and external D4 are excluded.

Full-volume outputs use `original_label_ids=True`. `postprocessing_settings` controls brain-mask cleanup and ventricle separation. Thresholds are voxel counts on the 0.1 mm grid; categorical ROI masks are not cleaned as binary masks. Merged D1/rat ventricles are separated before mapping to original IDs 3 (contra), 13 (ipsi), and 21 (third ventricle). MouseFull keeps its directly predicted sides. GT uses the selected original annotations, never the prediction postprocessor.

The historical `default_hemisphere="right"` selects the plane side containing array origin, rather than deriving anatomical right from the affine. It is used when no sufficiently large lesion is predicted. Splitting requires at least three predicted third-ventricle voxels; missing midline predictions raise an explicit scan-specific error.


`USE_TBI_PRETRAINING=True` prepares feature transfer from a previous TBI mouse run. Set `TBI_RUN` to its run folder and `TBI_FOLD` to its source fold. The loader reads `experiment.json` and selects the committed best-weight snapshot, so a completed epoch is sufficient even if training was interrupted. It reconstructs the four-class source graph and verifies backbone topology and weight shapes before transferring encoder/decoder features. The eleven-class ROI head, brain head and SHAM/TBI classifier start fresh; channels with different label meanings are not copied. With `FREEZE_BACKBONE=False`, all transferred features are fine-tuned. Architecture, loss and learning rate keep their existing settings.

No checkpoint is chosen automatically. With `TBI_RUN=None`, data previews work but training requires a source run. Set `USE_TBI_PRETRAINING=False` to deliberately train from scratch. Initialization details and source hashes are saved in each fold's `initialization.json`. `RESUME_RUN` restores the MouseFull fitting state rather than restarting from TBI weights.

TBI pretraining must exclude the animals used for MouseFull evaluation, including validation animals if they are intended as unseen checkpoint-selection data. MouseFull contains D1, so a general D1-trained checkpoint can overlap later MouseFull splits. For single training, align the target validation/test selection with animals unseen in source training and source selection-validation. For K-Fold, one shared checkpoint is valid only if it excludes every evaluated animal across those folds; otherwise use separately trained sources matching each fold's exclusions. This notebook's shared `TBI_RUN` is an explicit common initialization and does not automatically make a source split independent of the target folds.


In [ ]:
from typing import Literal

from src.experiments.config import (
    DataConfig, DatasetSpec, ExperimentConfig, InferenceConfig, InitializationConfig,
    LabelConfig, LossConfig, ModelConfig, PostprocessingConfig, PreprocessingConfig,
)
from src.evaluation.k_fold import EvaluationConfig, TrainingConfig

DATA_ROOT: Path = Path("D:/Other files/TESI_CNN/MANUSCRIPT_DATASET/normalized")
RUN_TRAINING: bool = False
USE_TBI_PRETRAINING: bool = True
TBI_RUN: Path | None = None  # Previous notebook 01/02 run folder, including interrupted runs.
TBI_FOLD: int = 0  # Source fold; single-run TBI training uses fold 0.
FREEZE_BACKBONE: bool = False  # False fine-tunes all transferred feature layers.
RESUME_RUN: Path | None = None
FOLD_FILE: Path | None = None
FOLD_MODE: Literal["random", "balanced"] = "balanced"
COHORT_SPLIT_FILE: Path | None = None

training = TrainingConfig(
    model_name="mouse_full_3drnet_kfold", k=5, epochs=300, batch_size=8,
    seed=TRAINING_SEED, data_multiplier=3, output_dir=str(REPO_ROOT / "results"),
    monitor="val_loss", monitor_mode="min", default_callbacks=False,
    fit_kwargs={"verbose": 1},
)
datasets = (DatasetSpec(name="D1", annotation_scope="full_volume"), DatasetSpec(name="D3", annotation_scope="full_volume"))
animal_ids: dict[str, str] = {}
data_settings = DataConfig(root=DATA_ROOT, datasets=datasets, animal_ids=animal_ids)
preprocessing = PreprocessingConfig(
    reference=REFERENCE, patch_size=(80, 80, 80), spacing_mm=(0.1, 0.1, 0.1),
    correct_units=True, augment=True, random_crop=True, crop_mode="center",
    # Native-grid MRI degradation; bounds retain their earlier magnitudes.
    intensity_augmentation_grid="native",
    blur_sigma_range=(0.0, 0.6), blur_sigma_units="voxels",
    noise_std_range=(0.0, 0.05), noise_std_units="intensity",
    flip_probability=0.0,
)

# Preserve the historical MouseFull channel order; keep each side separate.
label_mapping: dict[int, int] = {
    0: 0, 1: 1, 2: 2, 3: 3, 5: 4, 6: 5,
    12: 6, 13: 7, 15: 8, 16: 9, 21: 10,
}
class_names: tuple[str, ...] = (
    "Background", "Lesion", "Contra corpus callosum", "Contra ventricle",
    "Contra hippocampus", "Contra cortex", "Ipsi corpus callosum",
    "Ipsi ventricle", "Ipsi hippocampus", "Ipsi cortex", "Third ventricle",
)
labels = LabelConfig(mapping=label_mapping, names=class_names, unknown="background")
model_settings = ModelConfig(
    architecture="3drnet", filters=(16, 32, 64, 128, 256),
    residual=True, attention=True, max_pooling=True, classifier=True,
)
loss_settings = LossConfig(
    regions="dice_ce_gradient_fix", gradient_fix_version=2, gamma=0.8, brain_alpha=0.6, batch_wise=True,
)
# Full volumes: postprocess first, then restore original annotation IDs.
postprocessing_settings = PostprocessingConfig(
    clean_brain_mask=True,
    fill_small_holes=True, holes_max_area=20000,
    remove_small_objects=True, object_min_area=30000,
    split_ventricles=False, use_centroids=False,
    default_hemisphere="right", lesion_voxels_threshold=10,
)
inference_settings = InferenceConfig(
    enabled=True, stride=40, save_predictions=True,
    original_label_ids=True, postprocessing=postprocessing_settings,
)
# Transfer feature extraction/decoding, keeping all MouseFull output heads fresh.
initialization = InitializationConfig()
if USE_TBI_PRETRAINING:
    if TBI_RUN is not None:
        initialization = InitializationConfig.from_run(
            run_dir=TBI_RUN, fold=TBI_FOLD, freeze_backbone=FREEZE_BACKBONE,
        )
        print("TBI initialization weights:", initialization.checkpoint)
    else:
        print("Data preview only: set TBI_RUN before training, or disable USE_TBI_PRETRAINING for scratch.")

config = ExperimentConfig(
    data=data_settings, preprocessing=preprocessing, training=training,
    model=model_settings, labels=labels, loss=loss_settings,
    inference=inference_settings, initialization=initialization,
    learning_rate=0.001, split_seed=42,
    cv_validation_fraction=0.2,
)
config.validate()

## Configure callbacks directly

K-Fold uses only LR reduction, with no early stopping. Add TensorBoard/CSVLogger here if you want extra logging. Edit the callback constructors or the list in the next cell; the training call explicitly receives `callbacks=callbacks`. No callback is added by a hidden experiment factory. `TrainingConfig.default_callbacks=False` prevents duplicate automatic logging/LR callbacks. Checkpoint selection uses the visible `TrainingConfig.monitor` and `monitor_mode` settings.

Relative log paths resolve inside each run/fold, so folds and study variants do not share files. The runner copies unused callback instances for every fit and saves their settings in `callbacks.json`; resume requires the same saved settings. CSV logs append when resuming.

The runner always saves best weights and epoch recovery state, even when this list is empty. It restores the selected best checkpoint before evaluation. For that reason early stopping uses `restore_best_weights=False`: restoring weights inside its epoch-end hook would mix earlier weights with the latest optimizer in the recovery checkpoint. The final best-weight behavior matches the original workflow.


In [ ]:
import tensorflow as tf

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", mode="min", factor=0.5, patience=10,
    min_lr=1e-7, verbose=1,
)
callbacks: list[tf.keras.callbacks.Callback] = [reduce_lr]


## Grouped folds and annotation coverage

By default all registered D1+D3 animals enter five outer folds. `FOLD_MODE="balanced"` stratifies newly generated outer holdouts by complete versus partial **anatomical** annotation coverage. Complete requires every selected anatomical class in every scan of an animal; lesion annotation is excluded from this balance, so SHAM animals can count as complete. Availability comes from metadata, not nonzero mask values. Each group is spread approximately evenly across folds, preserving the cohort ratio without duplicating animals or forcing a 50/50 split.

`FOLD_MODE="random"` retains the original unstratified split. Inner checkpoint-selection validation remains random, and `cv_validation_fraction` reserves it from each remaining pool. Dataset, condition and modality are not stratified; review their balance separately. If a group has fewer animals than folds, some outer folds cannot contain it. Load a chosen `FOLD_FILE` for matched comparisons. Saved/resumed folds take priority over the generation mode and are not rebalanced.

Optionally set `COHORT_SPLIT_FILE` to notebook 05's saved `splits.json`. Its training and validation animals form the CV pool; its test animals remain a separate shared test cohort evaluated by each fold model. Those repeated predictions do not create additional biological samples. Coverage below describes the registered datasets; fold counts describe the selected CV pool.

`RESUME_RUN` loads the saved fold assignments. Retain the same configuration, identity overrides, source files and cohort selection. A supplied `FOLD_FILE` must agree with the resumed run. Separate outer holdouts are required in every fold.

In [ ]:
from IPython.display import display
from src.experiments.data import (
    SubjectSplit, fold_annotation_summary, fold_summary, make_cv_splits, prepare_data,
)
from src.evaluation.k_fold import load_fold_splits

prepared = prepare_data(config)
cv_ids = prepared.subjects.subject_ids
test_ids: tuple[str, ...] = ()

if COHORT_SPLIT_FILE is not None:
    cohort = SubjectSplit.load(COHORT_SPLIT_FILE, prepared.subjects)
    cv_ids = sorted((*cohort.train, *cohort.validation))
    test_ids = cohort.test

strata: dict[str, str] | None = None
if FOLD_MODE == "balanced":
    strata = prepared.annotation_groups(cv_ids, include_lesion=False)
elif FOLD_MODE != "random":
    raise ValueError("FOLD_MODE must be random or balanced")

evaluation = EvaluationConfig(
    test_ids=test_ids, evaluate_validation=False,
    save_predictions=config.inference.save_predictions,
)

if RESUME_RUN is not None:
    split_path = RESUME_RUN / "splits.json"
    folds = load_fold_splits(split_path, cv_ids, config.training.k)
    if FOLD_FILE is not None:
        requested_folds = load_fold_splits(FOLD_FILE, cv_ids, config.training.k)
        if requested_folds != folds:
            raise ValueError("FOLD_FILE differs from the resumed run's folds")
elif FOLD_FILE is not None:
    folds = load_fold_splits(FOLD_FILE, cv_ids, config.training.k)
else:
    folds = make_cv_splits(
        cv_ids, k=config.training.k, seed=config.split_seed,
        validation_fraction=config.cv_validation_fraction, strata=strata,
    )

if any(not fold.test_ids for fold in folds):
    raise ValueError("MouseFull K-Fold requires independent outer holdout animals")

summary = fold_summary(folds, prepared.subjects)
annotation_balance = fold_annotation_summary(folds, prepared, include_lesion=False)
manifest = prepared.subjects.manifest(cv_ids)
coverage = prepared.annotation_summary()
manifest_columns = ["Subject", "Dataset", "OriginalID", "Modality", "Scan"]
display(summary)
display(annotation_balance)
display(coverage)
display(manifest[manifest_columns].head())
print("Separate test animals:", len(test_ids))

## Check one deterministic validation batch

This preview checks the first fold without building a model. ROI, brain and condition targets remain separate. The displayed missing channels are unavailable annotations, not absent anatomical structures. Brain-mask supervision and condition labels remain available independently of ROI annotation coverage.

In [ ]:
import numpy as np

validation_generator = prepared.make_generator(folds[0].val_ids, training=False, batch_size=1)
images, targets = validation_generator[0]
print("Input:", images.shape)
print("ROI targets:", targets[0].shape)
print("Brain targets:", targets[1].shape)
print("Condition targets (SHAM=0, TBI=1):", targets[2].ravel())
print("Validation batches:", len(validation_generator))

unavailable = np.all(targets[0][0] == -1, axis=(0, 1, 2))
missing_names = np.asarray(config.labels.names)[unavailable]
print("Unannotated channels in this scan:", missing_names.tolist())

del images, targets, validation_generator

## Train or resume and evaluate outer volumes

The shared runner creates a fresh model per fold and restores validation-selected best weights before evaluating the outer holdout. Checkpoint-selection validation is not used as the final performance table. Full-volume metrics use physical spacing and retain unavailable/empty-mask statuses; every scan is predicted once per evaluation pass.

Predictions restore each input's original voxel grid and contain original annotation IDs after the explicitly configured postprocessing. The classifier contributes to fitting and is exported; this notebook reports anatomical segmentation measurements. Recovery restores the last committed epoch and skips completed folds. Interrupted epochs restart, and exact stochastic replay is not guaranteed.

In [ ]:
from src.experiments.training import run_cross_validation

result = None
if RUN_TRAINING:
    if USE_TBI_PRETRAINING and config.initialization.mode == "scratch":
        raise ValueError("Set TBI_RUN and rerun configuration, or set USE_TBI_PRETRAINING=False for scratch")

    result = run_cross_validation(
        prepared, subject_ids=cv_ids, folds=folds,
        evaluation=evaluation, resume_run=RESUME_RUN, callbacks=callbacks,
    )
    print("Run saved to:", result.run_dir)
else:
    print("Checks complete. Set RUN_TRAINING = True to train or resume.")

## Inspect independent outer results

Plotly uses the native VS Code renderer. The table selects `DatasetRole="holdout"`; any separate shared test cohort retains the `test` role. Inspect `Status` before averaging: unavailable or undefined measurements must not become zero scores. Status counts below count metric rows, not biological animals.

The saved run includes the complete configuration, label policy, animal manifest, folds, initialization audits, epoch recovery state, best weights, inference models, histories, timings, full-volume predictions and metric tables. Keep scan-to-animal grouping when performing manuscript statistics. Training stays disabled until `RUN_TRAINING=True`.

In [ ]:
import pandas as pd
import plotly.io as pio
from src.experiments.training import history_figure

pio.renderers.default = "vscode"

if result is not None:
    figure = history_figure(result)
    figure.show()
    display(pd.DataFrame(result.timings))

    if result.evaluations.get("regions"):
        metrics = pd.concat(result.evaluations["regions"], ignore_index=True)
        evaluated = metrics.loc[metrics["DatasetRole"] == "holdout"]
        dice = evaluated.loc[evaluated["Metric"] == "Dice"]
        columns = ["Fold", "Subject", "Scan", "Region", "Value", "Status"]
        display(dice[columns])
        print("Full metric tables:", result.run_dir / "regions")
        status_columns = ["Fold", "Region", "Status"]
        status_counts = evaluated.groupby(status_columns, dropna=False).size()
        display(status_counts.rename("Metric rows"))

## Explore outer holdout predictions against ground truth

Run the cell below after training/evaluation, or point `VIEW_RUN` at an existing matching run folder to inspect it without fitting. Saved full-volume `regions.nii.gz` files are loaded directly; no predictions are recomputed. This requires inference and prediction saving to have been enabled.

The table lists every available scan/timepoint and fold. The viewer initially loads up to three rows; set `VIEW_SCAN_INDICES=(0, 4)` to explore other rows. Use the Plotly dropdown to select a scan and reference-oriented plane, then drag the slice slider. Zoom/pan works in each MRI/GT/prediction panel.

Both overlays use the same class colors. The title lists unannotated GT regions: their absence is not evidence of a prediction error. Set `VIEW_REGION` to a foreground **original annotation ID** (for example 13 for the ipsilateral ventricle) to isolate one region. `VIEW_OPACITY` controls the overlay. Previews sample up to 64 slices per plane and 192 pixels per side; set `VIEW_MAX_SLICES=None` for every slice and raise `VIEW_MAX_PIXELS` for more detail. Saved masks are unchanged. The controls use native Plotly rendering.


In [ ]:
from src.evaluation.inference import saved_prediction_manifest, prediction_comparison_figure

VIEW_RUN: Path | None = None  # Existing run folder, or use the result above.
VIEW_SCAN_INDICES: tuple[int, ...] | None = None  # None: first three manifest rows.
VIEW_REGION: int | None = None  # Original annotation ID (e.g. 13 = ipsi ventricle); None shows all.
VIEW_OPACITY: float = 0.55
VIEW_MAX_SLICES: int | None = 64
VIEW_MAX_PIXELS: int = 192
VIEW_ROLE: str = 'holdout'

viewer_role = VIEW_ROLE
viewer_run = VIEW_RUN
current_result = globals().get("result")
if viewer_run is None and current_result is not None:
    viewer_run = current_result.run_dir
if viewer_run is None and RESUME_RUN is not None:
    viewer_run = RESUME_RUN

if viewer_run is None:
    print("Train/evaluate first, or set VIEW_RUN to an existing run folder.")
else:
    prediction_views = saved_prediction_manifest(prepared, viewer_run, role=viewer_role)
    display(prediction_views)

    if prediction_views.empty:
        print("No saved predictions for this role. Check the run, VIEW_ROLE and prediction-saving settings.")
    else:
        prediction_figure = prediction_comparison_figure(
            data=prepared, predictions=prediction_views,
            scan_indices=VIEW_SCAN_INDICES, region=VIEW_REGION,
            opacity=VIEW_OPACITY, max_slices=VIEW_MAX_SLICES,
            max_pixels=VIEW_MAX_PIXELS,
        )
        prediction_figure.show(renderer="vscode")


## Evaluate the best checkpoint after stopping training

Interrupt training, then run this cell independently of `result`. Set `CHECKPOINT_RUN` to the existing **run folder**, select `CHECKPOINT_FOLD`, and enable `RUN_CHECKPOINT_EVALUATION`. After a kernel restart, only the repository setup cell and this cell are needed: the saved experiment supplies architecture, preprocessing, label policy and animal selection.

The default cohort matches this notebook. Fixed-split runs use test animals, CV uses outer holdouts, and historical rat CV uses validation animals. Every selected animal includes all timepoints. A best checkpoint must already exist; an unfinished epoch is not evaluated. The committed selected-weight copy is used when recovery state is available.

This performs full-volume inference only, saves metrics/predictions to a fresh folder outside the training run, and opens the MRI/GT/prediction viewer. Training weights, optimizer, history and completion flags are left intact. Inference can take time; leave the switch False to skip it. Use notebook 09 to reopen an evaluation without recomputing predictions.

The common RARE reference is explicitly used for checkpoint evaluation, including older checkpoints. A model trained with a different orientation receives a changed input convention; those scores should be interpreted as evaluation with an orientation override. Training recovery files remain unchanged.


The independent cell exposes `checkpoint_postprocessing` and always restores original annotation IDs, including when the saved run predates these settings. It writes `prediction_labels.json`, so the viewer uses the exact exported IDs. `CHECKPOINT_REGION` selects an original annotation ID. Changing evaluation settings does not change training targets or weights.


In [ ]:
from src.experiments.config import PostprocessingConfig
from src.evaluation.inference import (
    best_checkpoint_settings, evaluate_saved_model,
    saved_prediction_manifest, prediction_comparison_figure,
)

RUN_CHECKPOINT_EVALUATION: bool = False
CHECKPOINT_RUN: Path | None = None  # results/<model>/<run>; required after interruption/restart.
CHECKPOINT_FOLD: int = 0
CHECKPOINT_OUTPUT: Path | None = None  # None creates a fresh evaluation folder.
CHECKPOINT_SCAN_INDICES: tuple[int, ...] | None = None
CHECKPOINT_REGION: int | None = None
CHECKPOINT_MAX_SLICES: int | None = 64
CHECKPOINT_ROLE: str = 'holdout'

checkpoint_run = CHECKPOINT_RUN
current_result = globals().get("result")
if checkpoint_run is None and current_result is not None:
    checkpoint_run = current_result.run_dir
if checkpoint_run is None:
    checkpoint_run = globals().get("RESUME_RUN")

checkpoint_postprocessing = PostprocessingConfig(
    clean_brain_mask=True,
    fill_small_holes=True, holes_max_area=20000,
    remove_small_objects=True, object_min_area=30000,
    split_ventricles=False, use_centroids=False,
    default_hemisphere="right", lesion_voxels_threshold=10,
)

checkpoint_evaluation = None
if RUN_CHECKPOINT_EVALUATION:
    if checkpoint_run is None:
        raise ValueError("Set CHECKPOINT_RUN to the stopped run folder first")
    checkpoint_settings = best_checkpoint_settings(
        run_dir=checkpoint_run, fold=CHECKPOINT_FOLD, role=CHECKPOINT_ROLE,
        output_dir=CHECKPOINT_OUTPUT, reference=REFERENCE,
        original_label_ids=True, postprocessing=checkpoint_postprocessing,
    )
    print("Selected weights:", checkpoint_settings.model_file)
    checkpoint_evaluation = evaluate_saved_model(checkpoint_settings)
    print("Evaluation saved to:", checkpoint_evaluation.output_dir)
    checkpoint_metrics = checkpoint_evaluation.metrics
    checkpoint_dice = checkpoint_metrics.loc[checkpoint_metrics["Metric"] == "Dice"]
    display(checkpoint_dice)
    checkpoint_views = saved_prediction_manifest(
        data=checkpoint_evaluation.data, run_dir=checkpoint_evaluation.output_dir,
        role=CHECKPOINT_ROLE,
    )
    display(checkpoint_views)
    checkpoint_figure = prediction_comparison_figure(
        data=checkpoint_evaluation.data, predictions=checkpoint_views,
        scan_indices=CHECKPOINT_SCAN_INDICES, region=CHECKPOINT_REGION,
        max_slices=CHECKPOINT_MAX_SLICES,
    )
    checkpoint_figure.show(renderer="vscode")
else:
    print("Set CHECKPOINT_RUN and RUN_CHECKPOINT_EVALUATION=True to evaluate saved best weights.")
